# Data Collection

Runs the scrapers that produce the raw CSVs in `data/`. The scraping code is a tested package in `src/story_reception/scrape/`. This notebook only runs it.

| Module | What it does | Output |
|---|---|---|
| `scrape/base.py` | Shared config, sync and async HTTP clients (retry with exponential backoff, rotating user agents, `Retry-After` handling, a global pause after a 429), streaming `CsvWriter`, JSON listing cache, and the `BaseDiscoverer` / `BaseDetailScraper` / `BasePipeline` interfaces | |
| `scrape/goodreads.py` | Phase 1a: genre-tagged Listopia index pages (sync). Phase 1b: list pages (async, feeding a queue). Phase 2: book pages, parsing the embedded `__NEXT_DATA__` Apollo cache for metadata and reviews | `goodreads-books.csv`, `goodreads-reviews.csv` |
| `scrape/rottentomatoes.py` | Phase 1: Playwright scrolls 564 browse pages (rating × genre × sort) on a background thread. Phase 2: async movie, critic-review and audience-review pages | `rotten-tomatoes-movies.csv`, `rotten-tomatoes-critic-reviews.csv`, `rotten-tomatoes-audience-reviews.csv` |
| `scrape/__init__.py` | Registry and `run_pipelines()`. Each source is imported only when it runs | |

**Resumable:** discovered listings are cached to JSON, and the CSV writer appends, so a crashed run picks up where it stopped.

**Adding a source:** write a module with a config dataclass and the three subclasses, then add it to `PIPELINES` in `scrape/__init__.py`.

**Before scraping:** run locally (Playwright needs a real browser: `playwright install chromium`), keep the default delays and concurrency, and check each site's terms of use. The scraped data is not committed to this repo.

From a terminal, the same thing is `python -m story_reception.scrape goodreads rottentomatoes --output-dir data`.

In [1]:
import nest_asyncio

from story_reception.scrape import PIPELINES, run_pipelines  # noqa: F401 (used in the cell below)

nest_asyncio.apply()  # Jupyter already runs an event loop; this lets the pipelines call asyncio.run() inside it
print('Available sources:', ', '.join(PIPELINES))

Available sources: goodreads, rottentomatoes


In [2]:
# Uncomment one line to scrape. Each run can take hours; results stream to data/ as they arrive.

# run_pipelines(['goodreads'])
# run_pipelines(['rottentomatoes'])
# run_pipelines()                     # all sources